In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the exact count. If you tell me your database type and the table name (and any filters you want), I can give you the exact command and help interpret the result.

Common options:

- SQL databases (MySQL, PostgreSQL, SQLite, SQL Server)
  - Basic count: SELECT COUNT(*) AS total_artists FROM artists;
  - Count unique IDs (in case there are duplicates): SELECT COUNT(DISTINCT artist_id) AS total_artists FROM artists;
  - With a filter (e.g., not deleted): SELECT COUNT(*) AS total_artists FROM artists WHERE is_deleted = 0;

- MongoDB
  - db.artists.countDocuments({});
  - If you want distinct by a field: db.artists.distinct("artist_id").length

- Elasticsearch
  - GET /artists/_count

If you share the exact DB type and table/collection name (and any filters), I’ll provide the precise query and steps to run it.


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

There are 275 artists in the database.
